# Hindsight Quickstart — Customer Support Example

This mirrors the structure of Hindsight's own
[quickstart notebook](https://github.com/vectorize-io/hindsight-cookbook/blob/main/notebooks/01-quickstart.ipynb),
with our own example: a customer support assistant remembering facts about
customers across separate conversations.

- **Retain**: store information in memory
- **Recall**: retrieve memories matching a query
- **Reflect**: generate an answer by reasoning over stored memories

Part 2 goes one step further, into a pattern Hindsight's own
[per-user-memory](https://github.com/vectorize-io/hindsight-cookbook/blob/main/notebooks/02-per-user-memory.ipynb)
example uses: **one bank per customer**, instead of the single shared bank
from Part 1. That's the realistic setup — Part 1 uses one bank for
everybody purely to keep the basics simple.

## Prerequisites

Hindsight must already be running (see the main [README](../README.md)):

```bash
docker compose -f ../docker-compose.hindsight.yml up -d
```

## Installation

In [ ]:
%pip install --quiet hindsight-client nest_asyncio

## Connect to Hindsight

In [ ]:
# Jupyter already runs its own asyncio event loop; the Hindsight client uses
# run_until_complete() internally, and Python doesn't allow nested event loops
# by default. nest_asyncio patches this so the client works inside a notebook.
import nest_asyncio
nest_asyncio.apply()

from hindsight_client import Hindsight

HINDSIGHT_API_URL = "http://localhost:8888"
HINDSIGHT_UI_URL = "http://localhost:9999"
BANK_ID = "quickstart-demo"

client = Hindsight(base_url=HINDSIGHT_API_URL)

# Fresh start, in case this notebook has been run before.
try:
    client.delete_bank(BANK_ID)
except Exception:
    pass

## Part 1 — Quickstart

## Retain: Store Information

`retain` pushes new information into memory. Behind the scenes, an LLM extracts
structured facts, entities, and timing from the text you give it.

In [ ]:
client.retain(
    bank_id=BANK_ID,
    content="Ahmet Yilmaz, kurumsal hesabinda odeme yontemini kredi kartindan banka havalesine degistirdi.",
    context="odeme yontemi guncellemesi",
)

print(f"Dokumanlari gorebilirsin: {HINDSIGHT_UI_URL}/banks/{BANK_ID}?view=documents")

In [ ]:
# context ve timestamp ile bir kayit daha
from datetime import datetime, timezone

client.retain(
    bank_id=BANK_ID,
    content="Ahmet, aboneligini aylik plandan yillik plana yukseltti.",
    context="plan degisikligi",
    timestamp=datetime.now(timezone.utc),
)

## Recall: Retrieve Memories

`recall` retrieves memories matching a query. It searches in parallel by
meaning, keywords, entity/temporal links, and time range.

In [ ]:
results = client.recall(bank_id=BANK_ID, query="Ahmet odemeyi nasil yapiyor?")

print("Bulunanlar:")
for r in results.results:
    print(f"  - {r.text}")

In [ ]:
# Zamanla ilgili bir soru
results = client.recall(bank_id=BANK_ID, query="Bu hafta Ahmet'in hesabinda ne degisti?")

print("Bulunanlar:")
for r in results.results:
    print(f"  - {r.text}")

## Reflect: Generate an Answer

`reflect` goes further than `recall` — instead of returning raw matching facts,
it reasons over what's stored and writes an answer to your question.

In [ ]:
response = client.reflect(
    bank_id=BANK_ID,
    query="Destek ekibinin Ahmet hakkinda bilmesi gereken en onemli sey nedir?",
)
print(response.text)

## Cleanup

Delete the bank created during Part 1.

In [ ]:
client.delete_bank(BANK_ID)
print("Bank silindi.")

---

## Part 2 — Per-User Memory

Part 1 put every customer in one bank (`quickstart-demo`). That's fine for
learning the basics, but it doesn't hold up in production: nothing stops one
customer's `recall` from surfacing another customer's information.

The fix Hindsight's own cookbook uses: **one bank per customer.** Complete
isolation, and a simpler mental model than filtering a shared bank by
customer ID on every query.

In [ ]:
client.create_bank(bank_id="support-ahmet", name="Ahmet Yilmaz")
client.create_bank(bank_id="support-elif", name="Elif Kaya")

client.retain(
    bank_id="support-ahmet",
    content="Ahmet Yilmaz kurumsal hesap kullaniyor, odeme yontemi banka havalesi.",
)
client.retain(
    bank_id="support-elif",
    content="Elif Kaya bireysel hesap kullaniyor, iletisimde telefonu tercih ediyor.",
)

### Proof of Isolation

`recall` always ranks and returns its best-effort matches from the bank it's
given — it doesn't return an empty list just because nothing is truly
relevant. So the real isolation check isn't "how many results came back,"
it's "does Elif's own data ever show up in Ahmet's bank." It shouldn't,
no matter how the query is worded.

In [ ]:
result = client.recall(bank_id="support-ahmet", query="Elif hakkinda ne biliyoruz?")

# The real proof isn't the result count -- recall always returns its closest
# matches, even weak ones. The proof is that none of them are Elif's data.
leaked = [r.text for r in result.results if "Elif" in r.text]

print(f"support-ahmet bank'inda {len(result.results)} sonuc bulundu (bunlar Ahmet'in kendi kayitlari, asagida gorulebilir):")
for r in result.results:
    print(f"  - {r.text}")
print(f"\nBunlarin icinde Elif'e ait veri var mi? {bool(leaked)}")

### Updating a Conversation with `document_id`

A customer rarely contacts support just once. Retaining every message as a
separate memory would leave you with a pile of fragments instead of one
coherent conversation. Pass the same `document_id` on every `retain` call for
that conversation, and Hindsight replaces the previous version instead of
adding a duplicate.

In [ ]:
CONVERSATION_ID = "ahmet-kargo-sikayeti"

client.retain(
    bank_id="support-ahmet",
    content="Musteri: Kargom hala gelmedi.\nTemsilci: Kargo numaranizi kontrol ediyorum.",
    document_id=CONVERSATION_ID,
)

# Ayni musteri birkac dakika sonra tekrar yaziyor. Ayni document_id, yeni bir
# kayit degil -- konusma guncelleniyor.
client.retain(
    bank_id="support-ahmet",
    content=(
        "Musteri: Kargom hala gelmedi.\n"
        "Temsilci: Kargo numaranizi kontrol ediyorum.\n"
        "Musteri: Tesekkurler, ne zaman gelir?\n"
        "Temsilci: Yarina kadar teslim edilecek."
    ),
    document_id=CONVERSATION_ID,
)

result = client.recall(bank_id="support-ahmet", query="Kargo ne zaman gelecek?")
print("Bulunanlar:")
for r in result.results:
    print(f"  - {r.text}")

print(f"\nTek dokuman olarak gorebilirsin: {HINDSIGHT_UI_URL}/banks/support-ahmet?view=documents")

Notice the results aren't limited to the cargo conversation -- the payment
fact from the isolation check above shows up too, ranked lower. `recall`
searches everything in the bank, not just one document. That's expected:
it's a broad retrieval step, meant to be narrowed by a specific query or
handed to `reflect` for a synthesized answer, not treated as an exact
lookup.

## Cleanup

In [ ]:
client.delete_bank("support-ahmet")
client.delete_bank("support-elif")
client.close()
print("Banklar silindi, baglanti kapatildi.")